In [ ]:
from huggingface_hub import login
from datasets import load_dataset
from datasets import Dataset
from huggingface_hub import login, HfApi
import pandas as pd


import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety reasons

from huggingface_hub import login
login(os.environ["HF_TOKEN"])

synthetic_dataset = "businessrules/final_dataset_review"


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
ds = load_dataset(synthetic_dataset)
dataset = ds["train"]
dataset
dataset_test = dataset.to_pandas()
print(dataset_test.columns)

Index(['id', 'cd', 'br', 'Module', 'code lang'], dtype='object')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer

gold_path = "/content/drive/MyDrive/gold_manual_label_set2.csv"
df_gold = pd.read_csv(gold_path)

print(df_gold.columns)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

X_emb = model.encode(df_gold["br"].tolist(), normalize_embeddings=True)

y = df_gold["label"].tolist()   


Index(['id', 'br', 'similarity_to_no_rule_sentence_max', 'sim_bin', 'label'], dtype='object')


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_emb, y, test_size=0.35, random_state=42, stratify=y
)

# Train classifier
clf = LogisticRegression(max_iter=500)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred))


              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       1.00      0.81      0.90        16

    accuracy                           0.94        51
   macro avg       0.96      0.91      0.93        51
weighted avg       0.95      0.94      0.94        51



In [ ]:
def classify_subset_by_ids(df, id_list, id_col="ID", text_col="business_rule"):
    df[id_col] = df[id_col].astype(str)
    id_list = [str(x) for x in id_list]

    subset = df[df[id_col].isin(id_list)]

    if subset.empty:
        print("No matching IDs found.")
        return []

    # Extract text
    texts = subset[text_col].astype(str).tolist()

    X_emb = model.encode(texts, normalize_embeddings=True)

    preds = clf.predict(X_emb)

    label_map = {0: "NO_RULE", 1: "HAS_RULE"}

    results = []
    for (idx, row), pred in zip(subset.iterrows(), preds):
        results.append({
            "id": row[id_col],
            "label": int(pred),
            "label_name": label_map[int(pred)],
            "text_preview": row[text_col][:200]
        })

    return results


In [ ]:
import joblib
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download

# Load classifier
clf_path = hf_hub_download(
    repo_id="businessrules/br-classifier-logreg",
    filename="logreg_classifier.joblib"
)
clf = joblib.load(clf_path)

# Load encoder from HF
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
import joblib
import os

save_dir = "/content/drive/MyDrive/br_classifier_model"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(clf, f"{save_dir}/logreg_classifier.joblib")

import json
json.dump({"labels": ["no_rule", "has_rule"]}, open(f"{save_dir}/meta.json", "w"))

print("Model saved!")


Model saved!


In [ ]:
from huggingface_hub import create_repo

repo_id = "businessrules/br-classifier-logreg"

create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True  
)



RepoUrl('https://huggingface.co/businessrules/br-classifier-logreg', endpoint='https://huggingface.co', repo_type='model', repo_id='businessrules/br-classifier-logreg')

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/drive/MyDrive/br_classifier_model",
    repo_id="businessrules/br-classifier-logreg",
    repo_type="model"
)


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Original classifier
clf = LogisticRegression(max_iter=500)

# Stratified 5-fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# F1 scores
scores_f1 = cross_val_score(clf, X_emb, y, cv=skf, scoring='f1')

# Precision & Recall
scores_precision = cross_val_score(clf, X_emb, y, cv=skf, scoring='precision')
scores_recall = cross_val_score(clf, X_emb, y, cv=skf, scoring='recall')

# Results
print("=== 5-Fold Cross Validation (Original Model) ===")
print("F1 scores:      ", scores_f1)
print("Precision:      ", scores_precision)
print("Recall:         ", scores_recall)
print("\nMean F1:        ", scores_f1.mean())
print("Mean Precision: ", scores_precision.mean())
print("Mean Recall:    ", scores_recall.mean())


=== 5-Fold Cross Validation (Original Model) ===
F1 scores:       [0.875      0.875      1.         0.94117647 0.94117647]
Precision:       [1. 1. 1. 1. 1.]
Recall:          [0.77777778 0.77777778 1.         0.88888889 0.88888889]

Mean F1:         0.9264705882352942
Mean Precision:  1.0
Mean Recall:     0.8666666666666666


In [ ]:
## For finsing the tricky edge cases

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tricky_examples = []

for fold_num, (train_index, test_index) in enumerate(skf.split(X_emb, y), 1):
    X_train, X_test = X_emb[train_index], X_emb[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    clf = LogisticRegression(max_iter=500)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # Collect tricky NO_RULEs: misclassified or borderline
    for idx, (true_label, pred_label) in zip(test_index, zip(y_test, y_pred)):
        if true_label == 1:  # Only look at NO_RULEs
            if pred_label == 0:  # Misclassified
                tricky_examples.append((df.iloc[idx]["id"], "MISSED", df.iloc[idx]["br"][:200]))
            else:
                # Optionally, define borderline by model confidence if using predict_proba
                pass

# Remove duplicates (a NO_RULE might appear in multiple folds)
tricky_examples = list({ex[0]: ex for ex in tricky_examples}.values())

print(f"Total tricky NO_RULE examples: {len(tricky_examples)}")
for ex in tricky_examples[:10]:  # show first 10
    print(f"ID: {ex[0]}, Type: {ex[1]}, Preview: {ex[2]}...\n")


Total tricky NO_RULE examples: 10
ID: 771, Type: MISSED, Preview: # Business Rules for Booking Retrieval by Unit

This section does not contain any business rules. The code implements technical data retrieval logic without expressing business policy or domain rules....

ID: 642, Type: MISSED, Preview: # Business Rules for API Requests

These are implementation details rather than business-level policies.They describe technical HTTP behaviors such as request methods, URIs, headers, responses, loggin...

ID: 298, Type: MISSED, Preview: # Business Rules for Payment Term Group Condition

*No explicit business rules are defined at the business policy level in this section. The code provided only configures form structure and data mappi...

ID: 823, Type: MISSED, Preview: # Business Rules for Unit Type Rate Log Collection

*No business rules are defined for this section.*...

ID: 853, Type: MISSED, Preview: # Business Rules for Unit Type Rate Log Line

*No business rules are defined for this

In [ ]:
# Example: which IDs you want to classify
target_ids = [1,2,3,4,5,6,728]  

from datasets import load_dataset

# Load your full HF dataset
ds = load_dataset(synthetic_dataset)["train"]
df_full = ds.to_pandas()

# Filter by ID
df_target = df_full[df_full["id"].isin(target_ids)].copy()

# Embed BR texts
X_target = model.encode(df_target["br"].tolist(), normalize_embeddings=True)

# Predict
df_target["predicted_rule_label"] = clf.predict(X_target)
df_target["prediction_probability"] = clf.predict_proba(X_target).max(axis=1)

print(df_target[["id", "predicted_rule_label", "prediction_probability"]])
